# CUAD Data Exploration

Purpose:
1. List all CUAD question categories and their counts
2. Per-category count of contracts that have at least one gold span
3. Verify our 10-clause mapping covers real CUAD category strings
4. Validate prepared data splits

If Cell 3 prints missing categories, update `CUAD_TO_CLAUSE_TYPE` in
`src/data/prepare.py` to use the correct CUAD category strings, then
re-run `python -m src.data.prepare`.

In [ ]:
import collections

from src.data.prepare import _download_cuad_json

raw = _download_cuad_json()
rows = []
for doc in raw["data"]:
    cid = doc["title"]
    for para in doc["paragraphs"]:
        for qa in para["qas"]:
            rows.append({"id": cid, "question": qa["question"], "answers": qa.get("answers", [])})

print(f"Total QA rows: {len(rows)}")

cats = collections.Counter()
for row in rows:
    q = row["question"]
    cat = q.split('"')[1] if '"' in q else q
    cats[cat] += 1

print(f"\n{len(cats)} unique question categories:\n")
for cat, n in cats.most_common():
    print(f"{n:6d}  {cat}")

In [ ]:
cat_to_contracts_with_span = collections.defaultdict(set)
for row in rows:
    if len(row["answers"]) > 0:
        cid = row["id"]
        cat = row["question"].split('"')[1] if '"' in row["question"] else row["question"]
        cat_to_contracts_with_span[cat].add(cid)

print("contracts-with-span count per category (desc):\n")
for cat, contracts in sorted(cat_to_contracts_with_span.items(), key=lambda kv: -len(kv[1])):
    print(f"{len(contracts):4d}  {cat}")

In [ ]:
from src.data.prepare import CUAD_TO_CLAUSE_TYPE

missing = [c for c in CUAD_TO_CLAUSE_TYPE if c not in cats]
if missing:
    print("CUAD categories referenced in our map but NOT FOUND in CUAD:")
    for m in missing:
        print(f"  - {m!r}")
    print(
        "\nFix CUAD_TO_CLAUSE_TYPE in src/data/prepare.py, then re-run `python -m src.data.prepare`."
    )
else:
    print("All 10 mapped categories found in CUAD. Mapping is valid.")

In [ ]:
from pathlib import Path

from src.data.validate import validate_jsonl

for split in ("train", "val", "test"):
    p = Path("data/processed") / f"{split}.jsonl"
    if not p.exists():
        print(f"[skip] {p} not found — run `python -m src.data.prepare` first")
        continue
    r = validate_jsonl(p)
    print(f"\n=== {split} ===")
    print(
        f"rows: {r.rows}  parse_err: {r.parse_errors}  schema_err: {r.schema_errors}  span_err: {r.span_errors}"
    )
    print(f"negative_ratio: {r.negative_ratio:.2%}")
    for ct, n in sorted(r.clause_counts.items()):
        marker = "  LOW" if n < 20 and split == "train" else ""
        print(f"  {ct}: {n}{marker}")